In [1]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
def route_distance(G, path):
    total = 0
    for i in range(len(path) - 1):
        total += G[path[i]][path[i + 1]]["weight"]
    return total

def route_cost(G, path, alpha=1.0, beta=100.0):
    distance = route_distance(G, path)
    hops = len(path) - 1
    return alpha * distance + beta * hops

In [3]:
def analyze_route(G, source, target, alpha=1.0, beta=100.0):
    if source not in G.nodes or target not in G.nodes:
        return {
            "source": source,
            "target": target,
            "path": "City not in graph",
            "distance": None,
            "hops": None,
            "cost": None
        }

    if not nx.has_path(G, source, target):
        return {
            "source": source,
            "target": target,
            "path": "No path",
            "distance": None,
            "hops": None,
            "cost": None
        }

    path = nx.shortest_path(G, source=source, target=target, weight="weight")
    distance = route_distance(G, path)
    hops = len(path) - 1
    cost = route_cost(G, path, alpha=alpha, beta=beta)

    return {
        "source": source,
        "target": target,
        "path": " -> ".join(path),
        "distance": round(distance, 2),
        "hops": hops,
        "cost": round(cost, 2)
    }

In [5]:
def compare_routes(G1, G2, pairs, alpha=1.0, beta=100.0, name1="Threshold", name2="KNN"):
    results = []

    for source, target in pairs:
        r1 = analyze_route(G1, source, target, alpha, beta)
        r2 = analyze_route(G2, source, target, alpha, beta)

        results.append({
            "graph": name1,
            "source": r1["source"],
            "target": r1["target"],
            "path": r1["path"],
            "distance": r1["distance"],
            "hops": r1["hops"],
            "cost": r1["cost"]
        })

        results.append({
            "graph": name2,
            "source": r2["source"],
            "target": r2["target"],
            "path": r2["path"],
            "distance": r2["distance"],
            "hops": r2["hops"],
            "cost": r2["cost"]
        })

    return pd.DataFrame(results)

In [4]:
def plot_route(G, path_list, title):
    pos = nx.get_node_attributes(G, "pos")
    sizes = [G.nodes[node]["population"] / 5000 for node in G.nodes()]

    plt.figure(figsize=(12, 8))

    nx.draw(
        G,
        pos,
        node_size=sizes,
        edge_color="gray",
        alpha=0.2,
        with_labels=False
    )

    path_edges = list(zip(path_list, path_list[1:]))

    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=path_list,
        node_size=[G.nodes[node]["population"] / 3000 for node in path_list]
    )

    nx.draw_networkx_edges(
        G,
        pos,
        edgelist=path_edges,
        width=2
    )

    labels = {node: node for node in path_list}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8)

    plt.xlim(-130, -65)
    plt.ylim(25, 50)
    plt.title(title)
    plt.show()

In [7]:
pairs = [
    ("Detroit", "Chicago"),
    ("Los Angeles", "San Francisco"),
    ("New York", "Boston"),
    ("Dallas", "Houston")
]

results_df = compare_routes(G_threshold, G_knn, pairs, alpha=1.0, beta=100.0)
print(results_df)

example = nx.shortest_path(G_knn, source="Detroit", target="Chicago", weight="weight")
plot_route(G_knn, example, "Efficient Route: Detroit to Chicago (KNN Graph)")

NameError: name 'G_threshold' is not defined